# Fine-tuning a symbol detector for architectural drawings

End-to-end: **synthetic dataset → document-level split → YOLOv8 fine-tune → mAP eval → tile-and-stitch inference → failure modes.**

This is the trained-detector counterpart to the live demo (plansight-henna.vercel.app). The demo shows the tile / stitch / seam-NMS engine with a classical region proposer; here the per-tile detector is a fine-tuned CNN that also *classifies* each symbol.

Runtime: set **Runtime → Change runtime type → T4 GPU**, then run all cells. Full run is a few minutes.

In [ ]:
!pip install -q ultralytics
!pip install -q --upgrade "numpy>=2.0,<2.3"
print("installed — now: Runtime -> Restart session, then Run all")

> **Important:** after this cell finishes, click **Runtime → Restart session**, then **Runtime → Run all**. The install shuffles numpy; the kernel must restart once to pick up the consistent version. The pin above makes it safe to re-run — the error will not come back.

In [ ]:
import os, math, random, glob
from dataclasses import dataclass
from PIL import Image, ImageDraw, ImageFont
import numpy as np

random.seed(0)
np.random.seed(0)

ROOT = "/content/arch"
IMG_DIR = os.path.join(ROOT, "images")
LBL_DIR = os.path.join(ROOT, "labels")
for d in [IMG_DIR, LBL_DIR]:
    os.makedirs(d, exist_ok=True)

CLASSES = ["door", "window", "fixture", "stairs", "north", "tree"]
CID = {c: i for i, c in enumerate(CLASSES)}
SHEET_W, SHEET_H = 1400, 1000
WALL = (150, 160, 172)
SYM = (17, 24, 39)

## 1 · Synthetic sheet generator

Each *document* is a full sheet with a random room layout and randomly placed symbols. Structural linework (walls, borders, dimensions) is drawn in a lighter weight than the symbol layer, mirroring real CAD line-weight conventions. Every symbol returns a class + pixel box, which becomes a ground-truth label.

In [ ]:
def _door(d, x, y, s):
    d.line([(x, y), (x, y + s)], fill=SYM, width=3)
    d.arc([x - s, y - s, x + s, y + s], 0, 90, fill=SYM, width=3)
    d.line([(x, y), (x + s, y)], fill=SYM, width=3)
    return ("door", x - 4, y - 4, x + s + 4, y + s + 4)

def _window(d, x, y, w):
    d.rectangle([x, y, x + w, y + 10], outline=SYM, width=3)
    d.line([(x, y + 5), (x + w, y + 5)], fill=SYM, width=3)
    return ("window", x - 3, y - 3, x + w + 3, y + 13)

def _fixture(d, x, y):
    d.rounded_rectangle([x, y, x + 34, y + 26], radius=8, outline=SYM, width=3)
    d.ellipse([x + 7, y + 6, x + 27, y + 22], outline=SYM, width=3)
    return ("fixture", x - 4, y - 4, x + 38, y + 30)

def _stairs(d, x, y):
    d.rectangle([x, y, x + 54, y + 70], outline=SYM, width=3)
    for i in range(1, 7):
        d.line([(x, y + i * 10), (x + 54, y + i * 10)], fill=SYM, width=2)
    d.line([(x + 27, y + 68), (x + 27, y + 2)], fill=SYM, width=3)
    return ("stairs", x - 4, y - 4, x + 58, y + 74)

def _north(d, x, y):
    d.ellipse([x - 22, y - 22, x + 22, y + 22], outline=SYM, width=3)
    d.line([(x, y + 16), (x, y - 16)], fill=SYM, width=3)
    d.line([(x, y - 16), (x - 7, y - 6)], fill=SYM, width=3)
    d.line([(x, y - 16), (x + 7, y - 6)], fill=SYM, width=3)
    return ("north", x - 26, y - 26, x + 26, y + 26)

def _tree(d, x, y):
    d.ellipse([x - 16, y - 16, x + 16, y + 16], outline=SYM, width=2)
    for a in range(8):
        an = a / 8 * 2 * math.pi
        d.line([(x + math.cos(an) * 8, y + math.sin(an) * 8),
                (x + math.cos(an) * 16, y + math.sin(an) * 16)], fill=SYM, width=2)
    return ("tree", x - 20, y - 20, x + 20, y + 20)

In [ ]:
def make_sheet(seed):
    rnd = random.Random(seed)
    img = Image.new("RGB", (SHEET_W, SHEET_H), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([24, 24, SHEET_W - 24, SHEET_H - 24], outline=WALL, width=2)

    cols = rnd.choice([2, 3])
    rows = rnd.choice([2, 3])
    boxes = []
    mx, my = 70, 70
    cw = (SHEET_W - 2 * mx) // cols
    ch = (SHEET_H - 2 * my) // rows
    for r in range(rows):
        for c in range(cols):
            rx = mx + c * cw + rnd.randint(0, 20)
            ry = my + r * ch + rnd.randint(0, 20)
            rw = cw - rnd.randint(30, 70)
            rh = ch - rnd.randint(30, 70)
            d.rectangle([rx, ry, rx + rw, ry + rh], outline=WALL, width=2)

            placers = rnd.sample(["door", "window", "fixture", "tree", "stairs"], k=rnd.randint(1, 3))
            for p in placers:
                px = rx + rnd.randint(20, max(21, rw - 60))
                py = ry + rnd.randint(20, max(21, rh - 70))
                if p == "door":
                    boxes.append(_door(d, px, py, rnd.choice([40, 44, 48])))
                elif p == "window":
                    boxes.append(_window(d, px, py, rnd.choice([70, 90, 100])))
                elif p == "fixture":
                    boxes.append(_fixture(d, px, py))
                elif p == "tree":
                    boxes.append(_tree(d, px + 20, py + 20))
                elif p == "stairs" and rw > 90 and rh > 100:
                    boxes.append(_stairs(d, px, py))

    boxes.append(_north(d, SHEET_W - 90, 90))
    for _ in range(rnd.randint(6, 12)):
        x = rnd.randint(90, SHEET_W - 200)
        y = rnd.randint(90, SHEET_H - 120)
        d.line([(x, y), (x + 60, y)], fill=(170, 178, 190), width=1)
    return img, boxes

In [ ]:
sample_img, sample_boxes = make_sheet(1)
print("symbols on sample sheet:", len(sample_boxes))
sample_img

## 2 · Tile the sheets and split **by document**

A 600 dpi sheet is too large to feed a detector whole, so each sheet is tiled into overlapping crops and every symbol box is remapped into tile coordinates (kept if the majority of the box lands inside the tile).

The split is the part that matters on domain data: **entire documents go to train or val — never individual tiles or symbols.** Tiles from one sheet share the same architect's line style, symbol spacing, and noise; if tiles from the same sheet appeared in both train and val, the model would be scored on style it already memorized, and mAP would be optimistic. Holding out whole documents measures generalization to an unseen drawing.

In [ ]:
TILE = 350
STRIDE = 280
N_DOCS = 44
VAL_DOCS = set(range(36, 44))

def tiles_of(w, h, t, s):
    xs = list(range(0, max(1, w - t) + 1, s))
    ys = list(range(0, max(1, h - t) + 1, s))
    if xs[-1] != w - t: xs.append(max(0, w - t))
    if ys[-1] != h - t: ys.append(max(0, h - t))
    return [(x, y) for y in ys for x in xs]

def clip_box(bx, tx, ty, t):
    cls, x0, y0, x1, y1 = bx
    ix0, iy0 = max(x0, tx), max(y0, ty)
    ix1, iy1 = min(x1, tx + t), min(y1, ty + t)
    if ix1 <= ix0 or iy1 <= iy0:
        return None
    inter = (ix1 - ix0) * (iy1 - iy0)
    area = (x1 - x0) * (y1 - y0)
    if inter < 0.55 * area:
        return None
    return cls, ix0 - tx, iy0 - ty, ix1 - tx, iy1 - ty

for split in ["train", "val"]:
    os.makedirs(f"{IMG_DIR}/{split}", exist_ok=True)
    os.makedirs(f"{LBL_DIR}/{split}", exist_ok=True)

counts = {"train": 0, "val": 0}
for doc in range(N_DOCS):
    split = "val" if doc in VAL_DOCS else "train"
    img, boxes = make_sheet(1000 + doc)
    for (tx, ty) in tiles_of(SHEET_W, SHEET_H, TILE, STRIDE):
        crop = img.crop((tx, ty, tx + TILE, ty + TILE))
        lines = []
        for bx in boxes:
            cb = clip_box(bx, tx, ty, TILE)
            if cb is None:
                continue
            cls, x0, y0, x1, y1 = cb
            cx = (x0 + x1) / 2 / TILE
            cy = (y0 + y1) / 2 / TILE
            bw = (x1 - x0) / TILE
            bh = (y1 - y0) / TILE
            lines.append(f"{CID[cls]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        if not lines:
            continue
        name = f"d{doc:03d}_{tx}_{ty}"
        crop.save(f"{IMG_DIR}/{split}/{name}.png")
        with open(f"{LBL_DIR}/{split}/{name}.txt", "w") as f:
            f.write("\n".join(lines))
        counts[split] += 1

print("train tiles:", counts["train"], "| val tiles:", counts["val"])
print("val documents held out:", sorted(VAL_DOCS))

In [ ]:
yaml = f"""path: {ROOT}
train: images/train
val: images/val
names:
"""
for i, c in enumerate(CLASSES):
    yaml += f"  {i}: {c}\n"
open(f"{ROOT}/data.yaml", "w").write(yaml)
print(yaml)

## 3 · Fine-tune YOLOv8

Start from the COCO-pretrained `yolov8n` backbone and fine-tune on the symbol tiles. Small imgsz because tiles are small and symbols are tiny.

In [ ]:
import os, torch
os.environ["WANDB_DISABLED"] = "true"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > T4 GPU")

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(data=f"{ROOT}/data.yaml", epochs=15, imgsz=320, batch=32,
            cache=True, patience=8, seed=0, verbose=False, plots=True)

## 4 · Evaluate — mAP overall and per class

mAP50-95 is the strict COCO metric; mAP50 is the lenient one. Per-class AP shows which symbols are learned well and which are confused — the failure signal to act on.

In [ ]:
metrics = model.val(data=f"{ROOT}/data.yaml", verbose=False)
print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}\n")
print("per-class AP50:")
for i, c in enumerate(CLASSES):
    try:
        print(f"  {c:9s} {metrics.box.ap50[i]:.3f}")
    except Exception:
        print(f"  {c:9s} n/a")

## 5 · Tile-and-stitch inference on a held-out sheet

The trained model only sees tiles. To read a full sheet, tile it, predict per tile, map every box back to page coordinates, and run **non-max suppression across the tile seams** to remove the double-detections created by overlap — the same reconciliation the live demo visualizes, now with a learned detector.

In [ ]:
def nms_global(dets, iou_thr=0.4):
    dets = sorted(dets, key=lambda d: -d[5])
    keep = []
    def iou(a, b):
        x1, y1 = max(a[0], b[0]), max(a[1], b[1])
        x2, y2 = min(a[2], b[2]), min(a[3], b[3])
        iw, ih = max(0, x2 - x1), max(0, y2 - y1)
        inter = iw * ih
        ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
        return inter / ua if ua > 0 else 0
    for d in dets:
        if all(iou(d, k) <= iou_thr or d[4] != k[4] for k in keep):
            keep.append(d)
    return keep

test_img, test_boxes = make_sheet(9999)
raw = []
for (tx, ty) in tiles_of(SHEET_W, SHEET_H, TILE, STRIDE):
    crop = test_img.crop((tx, ty, tx + TILE, ty + TILE))
    res = model.predict(crop, imgsz=350, conf=0.35, verbose=False)[0]
    for b in res.boxes:
        x0, y0, x1, y1 = b.xyxy[0].tolist()
        raw.append([x0 + tx, y0 + ty, x1 + tx, y1 + ty, int(b.cls), float(b.conf)])

final = nms_global(raw, 0.4)
print(f"raw tile detections: {len(raw)}  ->  after seam NMS: {len(final)}  |  ground truth: {len(test_boxes)}")

In [ ]:
from PIL import ImageDraw as ID
vis = test_img.copy()
dv = ID.Draw(vis)
palette = [(220,38,38),(37,99,235),(21,128,61),(234,88,12),(147,51,234),(13,148,136)]
for x0, y0, x1, y1, cls, conf in final:
    dv.rectangle([x0, y0, x1, y1], outline=palette[cls % 6], width=3)
    dv.text((x0, y0 - 12), f"{CLASSES[cls]} {conf:.2f}", fill=palette[cls % 6])
vis

## 6 · Failure modes

Cataloging where the model breaks is half the job. This compares the stitched prediction to ground truth and buckets the errors.

In [ ]:
def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, x2 - x1), max(0, y2 - y1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0

gt = [(x0, y0, x1, y1, CID[c]) for (c, x0, y0, x1, y1) in test_boxes]
matched = set()
tp = wrong_class = 0
for det in final:
    best, bi = 0, -1
    for i, g in enumerate(gt):
        if i in matched:
            continue
        v = iou(det, g)
        if v > best:
            best, bi = v, i
    if best >= 0.4 and bi >= 0:
        matched.add(bi)
        if det[4] == gt[bi][4]:
            tp += 1
        else:
            wrong_class += 1

fp = len(final) - tp - wrong_class
fn = len(gt) - len(matched)
print(f"true positives      : {tp}")
print(f"wrong class         : {wrong_class}")
print(f"false positives     : {fp}")
print(f"missed (false neg)  : {fn}\n")
missed_cls = {}
for i, g in enumerate(gt):
    if i not in matched:
        missed_cls[CLASSES[g[4]]] = missed_cls.get(CLASSES[g[4]], 0) + 1
print("misses by class:", missed_cls or "none")

### What to take from this

- **Document-level split** is why these mAP numbers mean something — val sheets 36–43 were never seen in any form during training.
- **Seam NMS** collapses the overlap-zone duplicates; raising tile overlap trades compute for fewer boundary misses on wide symbols (windows, stairs).
- **Next on real data:** swap the synthetic generator for annotated architect drawings (Label Studio / CVAT), keep the document-level split, add classes to the vocabulary iteratively, and tie each detected symbol back to the governing zoning / building code rule.

The same tile-and-stitch reconciliation runs live at **plansight-henna.vercel.app** — this notebook replaces the classical proposer there with a detector that names each symbol.